# 11 · Date-clustered reversion-slope tests (Colab, CPU only)

Recomputes the reversion-slope t-statistics for Kronos mini/small/base and Chronos with Driscoll-Kraay standard errors (the 29 stocks share dates, so the across-stock t overstates precision). Uses the forecasts already on Drive: no GPU, no model downloads, no secrets. Slope estimates are unchanged; only the standard errors change. ~10 min.

Runtime → Change runtime type → **CPU** is fine.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/kronos-agent'
CODE = f'{ROOT}/code'

# Unpack code from Drive; refreshes automatically if the copy there predates the date-clustered SEs.
import os, shutil, zipfile, sys
ZIP = f'{ROOT}/kronos-agent.zip'
PB = f'{CODE}/kronos_agent/phaseb.py'
if not os.path.exists(PB) or 'slope_dk' not in open(PB).read():
    assert os.path.exists(ZIP), f'upload the new kronos-agent.zip to {ROOT}/ first'
    shutil.rmtree(CODE, ignore_errors=True)
    zipfile.ZipFile(ZIP).extractall(CODE)
    assert 'slope_dk' in open(PB).read(), 'kronos-agent.zip on Drive is the old one: upload the new zip'

%pip install -q ccxt yfinance
%pip install -q -e "$CODE" --no-deps
if CODE not in sys.path:
    sys.path.insert(0, CODE)

import glob
EQ = ' '.join(sorted(os.path.relpath(p, CODE) for p in glob.glob(f'{CODE}/configs/kronos_b/*.yaml')))
SMALL = 'model_id=NeoQuasar/Kronos-small'
MINI = 'model_id=NeoQuasar/Kronos-mini;tokenizer_id=NeoQuasar/Kronos-Tokenizer-2k'
CHR = 'model_family=chronos;model_id=amazon/chronos-t5-small;amp=false'
print(len(EQ.split()), 'equity configs')

## Run (CPU, ~10 min) — paste the output back

If it prints `missing paths:`, paste that too.

In [ ]:
!cd $CODE && python scripts/run_phaseb.py --root $ROOT --config $EQ --out modelsize --run base64 "lookback=64" --run base256 "lookback=256" --run small64 "lookback=64;$SMALL" --run small256 "lookback=256;$SMALL" --run mini64 "lookback=64;$MINI" --run mini256 "lookback=256;$MINI" --run chronos64 "lookback=64;$CHR" --run chronos256 "lookback=256;$CHR"
!cd $CODE && python scripts/run_modelsize.py --root $ROOT